In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = '/content/tb-classifier'
REPO_URL = 'https://github.com/vorrjjard-2/tb-classifier.git'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATASET_NAME = 'tbx11k'  # canonical local name; must match data.root in the YAML
DRIVE_ZIP = f'/content/drive/MyDrive/datasets/zips/{DATASET_NAME}.zip'
DATA_DIR = Path(REPO_DIR) / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target = DATA_DIR / DATASET_NAME

if not target.is_dir():
    # If a prior partial run left an extracted dir under a different name, pick it up
    # instead of re-unzipping. Otherwise copy + unzip from Drive.
    existing = [p for p in DATA_DIR.iterdir() if p.is_dir()]
    if not existing:
        !cp {DRIVE_ZIP} {DATA_DIR}/
        !unzip -q {DATA_DIR}/{DATASET_NAME}.zip -d {DATA_DIR}/
        !rm {DATA_DIR}/{DATASET_NAME}.zip
        existing = [p for p in DATA_DIR.iterdir() if p.is_dir()]
    if len(existing) != 1:
        raise RuntimeError(
            f"Expected exactly one dataset dir under {DATA_DIR}, got: "
            f"{[p.name for p in existing]}. Clean up manually."
        )
    if existing[0].name != DATASET_NAME:
        existing[0].rename(target)
        print(f"Renamed {existing[0].name}/ -> {DATASET_NAME}/")

!ls {target} | head

In [ ]:
!pip install -q -e . --no-deps
!pip install -q lightning torchmetrics albumentationsx opencv-python-headless \
    pydicom pyyaml wandb tqdm pillow pandas scikit-learn tensorboard

## 5. W&B login

Paste your API key when prompted. If you'd rather skip W&B for the first run, jump straight to the `--fast-dev-run` cell below.

In [ ]:
import wandb
wandb.login()

In [ ]:
!python scripts/train.py --config experiments/configs/flipr_resnet18_colab.yaml --fast-dev-run

## 7. Real training run

Checkpoints land in `/content/drive/MyDrive/tb-classifier-runs/resnet18-colab-001/checkpoints/` (configured in `resnet18_colab.yaml`), so they survive runtime disconnects.

In [ ]:
!python scripts/train.py --config experiments/configs/flipr_resnet18_colab.yaml